## TF-IDF representation evaluation

In [1]:
from utils.ml import tokenize, load_data, data_dir, evaluate_clustering_model
import matplotlib.pyplot as plt

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 06:00:34 WARN Utils: Your hostname, DESKTOP-UQF5BSK, resolves to a loopback address: 127.0.1.1; using 172.24.225.163 instead (on interface eth0)
26/04/22 06:00:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/22 06:00:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = load_data(data_dir)
sequences, tokens = tokenize(df)

TOKENIZED DF:


+----+-----+---+-----+-------+----------+--------------------+
|year|month|day|order|country|session ID|            sequence|
+----+-----+---+-----+-------+----------+--------------------+
|2008|    4|  1|    9|     29|         1|[197, 149, 51, 91...|
|2008|    4|  1|   10|     29|         2|[87, 12, 215, 9, ...|
+----+-----+---+-----+-------+----------+--------------------+
only showing top 2 rows
TOKEN INFO: 
+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+---+
|page 1 (main category)|page 2 (clothing model)|colour|location|model photography|price|price 2|page| id|
+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+---+
|                     4|                    P51|     9|       5|                1|   28|      2|   3|  0|
|                     4|                    P57|     4|       1|                1|   43|      1|   4|  1|
+----------------------+-----------------------+------+

### Count Vectorizer + IDF

In [3]:
from pyspark.ml.feature import CountVectorizer, IDF
from pyspark.ml import Pipeline
vocab_size = tokens.count()
cv = CountVectorizer(vocabSize=vocab_size, inputCol="sequence", outputCol="bow")
idf = IDF(inputCol= "bow", outputCol="tf-idf")
stages = [cv, idf]
tf_idf = Pipeline(stages=stages).fit(sequences)
vectorized = tf_idf.transform(sequences)
vectorized.show(5)

+----+-----+---+-----+-------+----------+--------------------+--------------------+--------------------+
|year|month|day|order|country|session ID|            sequence|                 bow|              tf-idf|
+----+-----+---+-----+-------+----------+--------------------+--------------------+--------------------+
|2008|    4|  1|    9|     29|         1|[197, 149, 51, 91...|(218,[0,19,28,34,...|(218,[0,19,28,34,...|
|2008|    4|  1|   10|     29|         2|[87, 12, 215, 9, ...|(218,[3,8,14,31,5...|(218,[3,8,14,31,5...|
|2008|    4|  1|    6|     21|         3|[91, 54, 151, 142...|(218,[25,28,44,61...|(218,[25,28,44,61...|
|2008|    4|  1|    4|     21|         4| [153, 183, 54, 102]|(218,[61,78,111,1...|(218,[61,78,111,1...|
|2008|    4|  1|    1|      9|         5|               [128]|    (218,[73],[1.0])|(218,[73],[3.5430...|
+----+-----+---+-----+-------+----------+--------------------+--------------------+--------------------+
only showing top 5 rows


### Evaluation

In [ ]:
results = {"K-Means": [], "Bisecting KMeans": [], "GMM": []}
from pyspark.ml.clustering import BisectingKMeans, KMeans, GaussianMixture
results["K-Means"].append(evaluate_clustering_model(KMeans(maxIter=30), vectorized, range(2, 21), features_col="tf-idf"))
results["Bisecting KMeans"].append(evaluate_clustering_model(BisectingKMeans(maxIter=50), vectorized, range(2, 21), features_col="tf-idf"))
results["GMM"].append(evaluate_clustering_model(GaussianMixture(), vectorized, range(2, 21), features_col="tf-idf"))
fig, ax = plt.subplots(1, 3, figsize=(15, 10), sharex=False)
models = list(results.keys())
scores = ["Silhouette", "DB", "CH"]
for r in range(1):
    for m in models:
        ax[r, 0].plot(results[m][r]["N_K"], results[m][r]["silouhette"], label= m, linestyle = "--", marker = "o", markersize = 4)
        ax[r, 1].plot(results[m][r]["N_K"], results[m][r]["davies_bouldin"], label= m, linestyle = "--", marker = "o", markersize = 4)
        ax[r, 2].plot(results[m][r]["N_K"], results[m][r]["calinski_harabasz"], label= m, linestyle = "--", marker = "o", markersize = 4)
    for c in range(3):
        ax[r, c].set_xlabel("K")
        if r == 0:
            ax[r, c].set_title(scores[c])
        if c == 0:
            ax[r, c].set_ylabel(f"score")
        if c == 2:
            ax[r, c].legend()
plt.tight_layout()
plt.show()

ERROR:root:KeyboardInterrupt while sending command.                             
Traceback (most recent call last):
  File "/home/leonid/miniconda3/envs/bigdata_env/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/leonid/miniconda3/envs/bigdata_env/lib/python3.10/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/home/leonid/miniconda3/envs/bigdata_env/lib/python3.10/socket.py", line 717, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt
ERROR:py4j.clientserver:Exception occurred while shutting down connection
Traceback (most recent call last):
  File "/home/leonid/miniconda3/envs/bigdata_env/lib/python3.10/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/home/leonid/miniconda3/envs/bigdata_env/lib/python3.10/site-packages/py4j/clients